In [1]:
#Load Data 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np

housing_df = pd.read_csv('housing_data_with_clustered_neigh.csv')
housing_df['log_latestPrice'] = np.log(housing_df['latestPrice'])
housing_df = housing_df.drop(['lotSizeSqFt', 'livingAreaSqFt','avgSchoolRating', 'avgSchoolSize', 'yearsOld'], axis=1)




In [2]:
# Select numeric columns only (excluding the target itself)
numeric_features = housing_df.select_dtypes(include=['number']).columns.drop(['latestPrice','log_latestPrice'])

# Calculate correlations with the target
correlations = housing_df[numeric_features].corrwith(housing_df['log_latestPrice'])

# Filter features with |correlation| > 0.3
selected_features = correlations[correlations.abs() > 0.3].sort_values(ascending=False)

print("Selected features with |correlation| > 0.3:")
print(selected_features)



Selected features with |correlation| > 0.3:
log_livingAreaSqFt          0.555706
numOfBathrooms              0.488907
avgSchoolRating_sq          0.427133
log_lotSizeSqFt             0.405558
MedianStudentsPerTeacher    0.346561
numOfBedrooms               0.337405
dtype: float64


In [3]:
X = housing_df.drop(columns=['log_latestPrice', 'latestPrice'])
y = housing_df['log_latestPrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [12]:
X_train.columns

Index(['latitude', 'longitude', 'propertyTaxRate', 'garageSpaces',
       'hasAssociation', 'hasGarage', 'hasSpa', 'hasView', 'numOfPhotos',
       'numOfAccessibilityFeatures',
       ...
       'neigh_cluster_20', 'neigh_cluster_21', 'neigh_cluster_22',
       'neigh_cluster_3', 'neigh_cluster_4', 'neigh_cluster_5',
       'neigh_cluster_6', 'neigh_cluster_7', 'neigh_cluster_8',
       'neigh_cluster_9'],
      dtype='object', length=152)

In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_pred_log = model.predict(X_test)

# Log-scale metrics
mse_log = mean_squared_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

# Convert predictions and actual values back to dollar scale
y_pred_dollars = np.exp(y_pred_log)
y_test_dollars = np.exp(y_test)

# Dollar-scale metrics
mse_dollars = mean_squared_error(y_test_dollars, y_pred_dollars)
rmse_dollars = np.sqrt(mse_dollars)

# Print results
#     print(f"Features used: {feature_list}")
print(f"--- Log Scale ---")
print(f"Test MSE (log scale): {mse_log:.4f}")
print(f"Test R² (log scale): {r2_log:.4f}")
print(f"--- Dollar Scale ---")
print(f"Test MSE (dollars): ${mse_dollars:,.2f}")
print(f"Test RMSE (dollars): ${rmse_dollars:,.2f}")

--- Log Scale ---
Test MSE (log scale): 0.0608
Test R² (log scale): 0.7591
--- Dollar Scale ---
Test MSE (dollars): $32,056.81
Test RMSE (dollars): $179.04


In [5]:

# --- LassoCV ---
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)
y_pred_log = lasso.predict(X_test)
y_pred_dollars = np.exp(y_pred_log)
y_test_dollars = np.exp(y_test)
rmse_dollars = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

lasso_coefs = pd.Series(lasso.coef_, index=X.columns)
lasso_selected = lasso_coefs[lasso_coefs != 0]
print('Lasso Alpha: ',lasso.alpha_)
print('RMSE in$: ',rmse_dollars)
print('Selected Features: ',lasso_selected.sort_values(key=abs, ascending=False))

Lasso Alpha:  0.06829052829654203
RMSE in$:  295.45628138504634
Selected Features:  numOfBathrooms        0.155096
avgSchoolRating_sq    0.007902
numOfPhotos           0.002030
days_since_sale      -0.000176
yearsOld_sq           0.000068
dtype: float64


In [6]:
# --- RidgeCV ---
ridge = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge.fit(X_train, y_train)
y_pred_log = ridge.predict(X_test)
y_pred_dollars = np.exp(y_pred_log)
rmse_dollars = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

ridge_coefs = pd.Series(ridge.coef_, index=X.columns)
print('Ridge Alpha: ',ridge.alpha_)
print('RMSE in$: ',rmse_dollars)
print('Selected Features: ',ridge_coefs.sort_values(key=abs, ascending=False))

Ridge Alpha:  4.328761281083062
RMSE in$:  183.96283057720768
Selected Features:  neigh_cluster_14       -0.584866
zipcode78703            0.542186
log_livingAreaSqFt      0.512695
zipcode78704            0.501472
zipcode78754           -0.402039
                          ...   
numOfParkingFeatures   -0.000619
numOfAppliances         0.000399
numOfPhotos             0.000362
days_since_sale        -0.000154
yearsOld_sq            -0.000005
Length: 152, dtype: float64


In [7]:
# --- ElasticNetCV ---
elastic = ElasticNetCV(cv=5, random_state=42, l1_ratio=[.1, .5, .7, .9, .95, 1])
elastic.fit(X_train, y_train)
y_pred_log = elastic.predict(X_test)
y_pred_dollars = np.exp(y_pred_log)
rmse_dollars = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

elastic_coefs = pd.Series(elastic.coef_, index=X.columns)
elastic_selected = elastic_coefs[elastic_coefs != 0]

print('ElasticNet Alpha: ',elastic.alpha_)
print('RMSE in$: ',rmse_dollars)
print('Selected Features: ',elastic_selected.sort_values(key=abs, ascending=False))

ElasticNet Alpha:  0.06829052829654203
RMSE in$:  295.45628138504634
Selected Features:  numOfBathrooms        0.155096
avgSchoolRating_sq    0.007902
numOfPhotos           0.002030
days_since_sale      -0.000176
yearsOld_sq           0.000068
dtype: float64


In [8]:
#Select features from Lasso 

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = housing_df.drop(columns=['log_latestPrice', 'latestPrice']) 
y = housing_df['log_latestPrice']

pipeline = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42))
pipeline.fit(X, y)

# Get coefficients
coef = pipeline.named_steps['lassocv'].coef_
features = pd.Series(coef, index=X.columns)
selected = features[features != 0].sort_values(key=abs, ascending=False)
print("Selected features by Lasso:\n", selected)

Selected features by Lasso:
 log_livingAreaSqFt      0.197574
zipcode78704            0.102142
neigh_cluster_15       -0.100200
zipcode78703            0.083912
neigh_cluster_12       -0.081948
                          ...   
neigh_WEST CONGRESS    -0.000526
zipcode78717            0.000499
zipcode78750            0.000390
numOfParkingFeatures   -0.000275
numOfAppliances         0.000236
Length: 139, dtype: float64


In [9]:
selected_feature_names = selected.index.tolist()
unselected_feature_names = housing_df.columns.difference(selected_feature_names)
df_without_selected = housing_df[unselected_feature_names]
df_without_selected.columns

Index(['latestPrice', 'log_latestPrice', 'neigh_HANCOCK',
       'neigh_JOHNSTON TERRACE', 'neigh_MLK', 'neigh_NORTH LOOP',
       'neigh_SOUTH MANCHACA', 'neigh_UPPER BOGGY CREEK',
       'neigh_WEST UNIVERSITY', 'neigh_WINDSOR ROAD', 'zipcode78721',
       'zipcode78741', 'zipcode78745', 'zipcode78752', 'zipcode78759'],
      dtype='object')

In [10]:
#Run model on Lasso selected features 
def run_linear_model(feature_list, df, target_col='log_latestPrice', test_size=0.3, random_state=42):
    # Split into X and y
    X = df[feature_list]
    y = df[target_col]
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_log = model.predict(X_test)
    
    # Log-scale metrics
    mse_log = mean_squared_error(y_test, y_pred_log)
    r2_log = r2_score(y_test, y_pred_log)
    
    # Convert predictions and actual values back to dollar scale
    y_pred_dollars = np.exp(y_pred_log)
    y_test_dollars = np.exp(y_test)
    
    # Dollar-scale metrics
    mse_dollars = mean_squared_error(y_test_dollars, y_pred_dollars)
    rmse_dollars = np.sqrt(mse_dollars)
    
    # Print results
    print(f"--- Log Scale ---")
    print(f"Test MSE (log scale): {mse_log:.4f}")
    print(f"Test R² (log scale): {r2_log:.4f}")
    print(f"--- Dollar Scale ---")
    print(f"Test MSE (dollars): ${mse_dollars:,.2f}")
    print(f"Test RMSE (dollars): ${rmse_dollars:,.2f}")
    
    return model


run_linear_model(selected_feature_names, housing_df)

--- Log Scale ---
Test MSE (log scale): 0.0603
Test R² (log scale): 0.7611
--- Dollar Scale ---
Test MSE (dollars): $31,710.76
Test RMSE (dollars): $178.08


LinearRegression()

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
import numpy as np

# Initialize model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test set
y_pred_log = model.predict(X_test)

# --- Log-scale metrics on test set ---
mse_log = mean_squared_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

# Convert predictions back to dollars
y_pred_dollars = np.exp(y_pred_log)
y_test_dollars = np.exp(y_test)

# --- Dollar-scale metrics on test set ---
mse_dollars = mean_squared_error(y_test_dollars, y_pred_dollars)
rmse_dollars = np.sqrt(mse_dollars)

# --- Cross-validated RMSE on log scale ---
neg_mse_scores_log = cross_val_score(model, X_train, y_train,
                                      scoring='neg_mean_squared_error', cv=5)
rmse_cv_log = np.mean(np.sqrt(-neg_mse_scores_log))

# --- Cross-validated RMSE on dollar scale (custom) ---
# Get cross-validated predictions
cv_preds_log = cross_val_predict(model, X_train, y_train, cv=5)
cv_preds_dollars = np.exp(cv_preds_log)
y_train_dollars = np.exp(y_train)

# Compute RMSE in dollars
rmse_cv_dollars = np.sqrt(mean_squared_error(y_train_dollars, cv_preds_dollars))

# --- Print results ---
print(f"--- Test Set ---")
print(f"Test MSE (log scale): {mse_log:.4f}")
print(f"Test R² (log scale): {r2_log:.4f}")
print(f"Test RMSE (dollars): ${rmse_dollars:,.2f}")

print(f"\n--- Cross-Validation ---")
print(f"CV RMSE (log scale): {rmse_cv_log:.4f}")
print(f"CV RMSE (dollars): ${rmse_cv_dollars:,.2f}")


--- Test Set ---
Test MSE (log scale): 0.0608
Test R² (log scale): 0.7591
Test RMSE (dollars): $179.04

--- Cross-Validation ---
CV RMSE (log scale): 0.2849
CV RMSE (dollars): $179.52
